# eval_base：在 Colab 上评估原始 Qwen2.5-3B-Instruct（MedQA 前 50 题，调试用）

步骤：检查 GPU → 设置路径变量 → clone repo → 安装依赖 → 挂载 Google Drive → 跑单元测试 → 看数据样例 → 评估 → 算指标 → 存到 Drive。

**运行前**：菜单「代码执行程序 → 更改运行时类型」选择 GPU（T4 即可）。

In [ ]:
!nvidia-smi

## 1. 需要你修改的变量

In [ ]:
# ======== 按需修改 ========
REPO_URL = "https://github.com/skjdsk/medqa-calibration.git"   # 整个作业的 repo 地址
SUBDIR = "."   # 评估代码在仓库里的文件夹名（改了文件夹名就改这里）
DRIVE_OUT_DIR = "/content/drive/MyDrive/medqa-calibration-results"          # Drive 共享文件夹路径（共享文件夹请先在 Drive 里“添加快捷方式到我的云端硬盘”）
DATASET = "medqa"        # medqa / medmcqa / pubmedqa（可写多个，用空格隔开）
NUM_OPTIONS = 4          # MedQA 选项数：4 或 5
N_DEBUG = 50             # 只跑前 N 题；跑完整测试集时改成 None
ADAPTER = None           # 评估训练后的模型时填 LoRA adapter 路径，如 "/content/drive/MyDrive/adapters/sft_seed1"
MODEL_NAME = "base"      # 写进结果文件的模型名，如 "sft_seed1"
BATCH_SIZE = 8           # T4 上 8 一般没问题；PubMedQA 摘要较长，爆显存就改成 4
# ==========================
REPO_DIR = "/content/project"            # 仓库 clone 到 Colab 的位置
CODE_DIR = f"{REPO_DIR}/{SUBDIR}"        # 所有命令都在这个文件夹里运行
LOCAL_OUT = "/content/results"

## 2. 获取代码、安装依赖

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
# 进入评估代码所在的子文件夹：python -m eval.xxx 必须在这里运行，才能找到 eval 包
%cd {CODE_DIR}
!pip install -q -r requirements.txt


## 3. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
print("结果会保存到：", DRIVE_OUT_DIR)

## 4.（可选）登录 Hugging Face
匿名下载有频率限制；如果下载很慢或报 429 错误，在 Colab 左侧「钥匙」图标里添加名为 `HF_TOKEN` 的 secret 后运行本格。

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("已设置 HF_TOKEN")
except Exception as e:
    print("未设置 HF_TOKEN（可以忽略）：", type(e).__name__)

## 5. 单元测试：确认指标公式正确（几秒钟，不需要 GPU）

In [ ]:
!python -m pytest eval/test_metrics.py -q

## 6. 先看数据样例（不加载模型）
**请仔细核对**：字段名、选项是否完整、标准答案字母是否和答案文字对得上、答案分布是否合理。

In [ ]:
!python -m eval.data_loaders --dataset medqa --num_options {NUM_OPTIONS} --n 3

## 7. 评估
输出里会打印：字母对应的 token id、第一道题的完整 prompt、准确率、平均 letter_mass（应接近 1）。

In [ ]:
n_arg = f"--n {N_DEBUG}" if N_DEBUG else ""
adapter_arg = f'--adapter "{ADAPTER}"' if ADAPTER else ""
!python -m eval.evaluate --dataset {DATASET} --num_options {NUM_OPTIONS} {n_arg} {adapter_arg} \
    --model_name {MODEL_NAME} --batch_size {BATCH_SIZE} --out_dir {LOCAL_OUT} --check_batch_consistency

## 8. 计算汇总指标、画可靠性图（只读 CSV，不需要模型）

In [ ]:
!python -m eval.metrics --pred_csv "{LOCAL_OUT}/preds_*.csv" --out_dir {LOCAL_OUT}/metrics

## 9. 复制到 Drive，并显示结果
每个模型的结果放在 `DRIVE_OUT_DIR/模型名/` 下，避免互相覆盖。

In [ ]:
import shutil, glob
import pandas as pd
from IPython.display import Image, display

dest = os.path.join(DRIVE_OUT_DIR, MODEL_NAME)
shutil.copytree(LOCAL_OUT, dest, dirs_exist_ok=True)
print("已复制到：", dest)
for f in sorted(glob.glob(os.path.join(dest, "**", "*"), recursive=True)):
    print("  ", f)

display(pd.read_csv(os.path.join(LOCAL_OUT, "metrics", "summary.csv")).round(4))
for png in sorted(glob.glob(os.path.join(LOCAL_OUT, "metrics", "reliability_*.png"))):
    display(Image(png, width=420))

## 之后：汇总 7 个模型
所有模型都跑完后，可以把 Drive 里所有 CSV 一起算一张总表：
```
%cd {CODE_DIR}
!python -m eval.metrics --pred_csv "/content/drive/MyDrive/medqa-calibration-results/*/preds_*.csv" --out_dir /content/all_metrics
```